<a href="https://colab.research.google.com/github/Ehsan-Roohi/DSMC_Python/blob/main/Expansion_Wave_SBT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# --- DSMC CODE - V17.1 SBT COLLISION MODEL (OPTIMIZED) ---
# این نسخه مدل برخورد SBT را بر اساس الگوریتم بهینه استفانوف
# که توسط کاربر ارائه شد، اصلاح می‌کند. این روش بسیار سریع‌تر از
# نسخه ساده SBT است و هزینه محاسباتی O(N) دارد.

import numpy as np
import matplotlib.pyplot as plt
import numba
import time
from scipy.signal import savgol_filter
import pandas as pd

# ===================================================================
# ۱. بخش شبیه‌سازی (با تابع برخورد SBT بهینه‌شده)
# ===================================================================
MASS_AR = 39.948e-3 / 6.022e23; KB = 1.380649e-23

@numba.jit(nopython=True)
def calculate_vhs_cross_section_numba(vr_mag):
    # پارامترهای ثابت برای آرگون در این تابع تعریف شده‌اند
    d_ref = 4.17e-10
    t_ref = 273.0
    omega_vhs = 0.81
    mass_ar = 39.948e-3 / 6.022e23
    kb = 1.380649e-23

    if vr_mag < 1e-9: return 1e-30
    exponent = omega_vhs - 0.5
    c_ref_sq = 2 * kb * t_ref / mass_ar
    gamma_val = 1.04533
    d_sq = (d_ref**2) * ((c_ref_sq / vr_mag**2)**exponent) * (1 / gamma_val)
    return np.pi * d_sq

# --- تابع برخورد جدید بر اساس مدل SBT بهینه شده (روش استفانوف) ---
@numba.jit(nopython=True)
def perform_collisions_sbt_optimized(particles, indices_in_cell, cell_vol, dt, fnum):
    """
    برخوردها را با استفاده از الگوریتم بهینه SBT انجام می‌دهد.
    این روش تمام جفت‌ها را بررسی نمی‌کند و هزینه محاسباتی O(N) دارد.
    """
    num_particles_in_cell = len(indices_in_cell)
    if num_particles_in_cell < 2:
        return

    # حلقه روی ذره اول برخورد (i)
    for i in range(num_particles_in_cell - 1):
        p1_idx = indices_in_cell[i]

        # انتخاب تصادفی ذره دوم (j) از میان ذرات باقی‌مانده در لیست
        # این کار جایگزین حلقه تو در تو می‌شود.
        remaining_particles_count = num_particles_in_cell - (i + 1)
        j_offset = np.random.randint(0, remaining_particles_count)
        j = i + 1 + j_offset
        p2_idx = indices_in_cell[j]

        # محاسبه سرعت نسبی (g_ij در مقاله)
        vrx = particles[p1_idx, 1] - particles[p2_idx, 1]
        vry = particles[p1_idx, 2] - particles[p2_idx, 2]
        vrz = particles[p1_idx, 3] - particles[p2_idx, 3]
        vr_mag = np.sqrt(vrx**2 + vry**2 + vrz**2)

        if vr_mag < 1e-9:
            continue

        # محاسبه سطح مقطع برخورد (sigma_ij)
        sigma_t = calculate_vhs_cross_section_numba(vr_mag)

        # محاسبه احتمال برخورد وزن‌دار بر اساس فرمول (51)
        # ضریب وزنی (N'-i) در اینجا همان remaining_particles_count است
        weighting_factor = float(remaining_particles_count)
        collision_prob = (weighting_factor * fnum * dt * sigma_t * vr_mag) / cell_vol

        # اگر عدد تصادفی کمتر از احتمال برخورد بود، برخورد را انجام بده
        if np.random.rand() < collision_prob:
            # محاسبه سرعت مرکز جرم
            vcm_x = 0.5 * (particles[p1_idx, 1] + particles[p2_idx, 1])
            vcm_y = 0.5 * (particles[p1_idx, 2] + particles[p2_idx, 2])
            vcm_z = 0.5 * (particles[p1_idx, 3] + particles[p2_idx, 3])

            # انتخاب جهت جدید پس از برخورد
            cos_chi = 2 * np.random.rand() - 1.0
            sin_chi = np.sqrt(1.0 - cos_chi**2)
            phi_chi = 2.0 * np.pi * np.random.rand()

            # محاسبه بردار سرعت نسبی جدید
            vr_prime_x = vr_mag * sin_chi * np.cos(phi_chi)
            vr_prime_y = vr_mag * sin_chi * np.sin(phi_chi)
            vr_prime_z = vr_mag * cos_chi

            # به‌روزرسانی سرعت‌های ذرات پس از برخورد
            particles[p1_idx, 1:4] = vcm_x + 0.5 * vr_prime_x, vcm_y + 0.5 * vr_prime_y, vcm_z + 0.5 * vr_prime_z
            particles[p2_idx, 1:4] = vcm_x - 0.5 * vr_prime_x, vcm_y - 0.5 * vr_prime_y, vcm_z - 0.5 * vr_prime_z


def run_dsmc_simulation(sim_params):
    LX = sim_params['LX']; RHO_INIT = sim_params['RHO_INIT']; T_INIT = sim_params['T_INIT']
    NUM_CELLS_X = sim_params['NUM_CELLS_X']; PARTICLES_PER_CELL_INIT = sim_params['PARTICLES_PER_CELL_INIT']
    TOTAL_TIME = sim_params['TOTAL_TIME']; DT = sim_params['DT']
    MIN_PARTICLES_FOR_STATS = 20

    TOTAL_PARTICLES_SIM = (NUM_CELLS_X // 2) * PARTICLES_PER_CELL_INIT
    N_DENSITY_REAL = RHO_INIT / MASS_AR
    CELL_VOLUME_CONCEPTUAL = (LX / NUM_CELLS_X) * ((LX/10) * (LX/10))
    FNUM = (N_DENSITY_REAL * CELL_VOLUME_CONCEPTUAL) / PARTICLES_PER_CELL_INIT
    NUM_STEPS = int(TOTAL_TIME / DT); SAMPLING_INTERVAL = int(NUM_STEPS / 4)

    particles = initialize_gas_expansion(TOTAL_PARTICLES_SIM, NUM_CELLS_X, LX, PARTICLES_PER_CELL_INIT, T_INIT)

    results_history = {}
    cell_width = LX / NUM_CELLS_X

    results_history[0.0] = sample_properties(particles, NUM_CELLS_X, cell_width, FNUM, CELL_VOLUME_CONCEPTUAL, MIN_PARTICLES_FOR_STATS)

    for step in range(1, NUM_STEPS + 1):
        particles[:, 0] += particles[:, 1] * DT

        hit_left = particles[:, 0] < 0; particles[hit_left, 1] *= -1; particles[hit_left, 0] *= -1
        hit_right = particles[:, 0] > LX; particles[hit_right, 1] *= -1; particles[hit_right, 0] = 2 * LX - particles[hit_right, 0]

        cell_indices = (particles[:, 0] / cell_width).astype(np.int64)
        cell_indices = np.clip(cell_indices, 0, NUM_CELLS_X - 1)
        sorted_particle_indices = np.argsort(cell_indices)
        cell_counts = np.bincount(cell_indices, minlength=NUM_CELLS_X)
        cell_start_indices = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(cell_counts[:-1])))

        for i in range(NUM_CELLS_X):
            start = cell_start_indices[i]; end = start + cell_counts[i]
            indices_in_cell_i = sorted_particle_indices[start:end]

            # --- فراخوانی تابع برخورد SBT بهینه‌شده ---
            perform_collisions_sbt_optimized(particles, indices_in_cell_i, CELL_VOLUME_CONCEPTUAL, DT, FNUM)

        if step % SAMPLING_INTERVAL == 0 or step == NUM_STEPS:
            results_history[step * DT] = sample_properties(particles, NUM_CELLS_X, cell_width, FNUM, CELL_VOLUME_CONCEPTUAL, MIN_PARTICLES_FOR_STATS)

    return results_history

# ... (سایر توابع initialize_gas_expansion, sample_properties و بخش رسم نمودار بدون تغییر باقی می‌مانند) ...

def initialize_gas_expansion(total_particles, num_cells, lx, ppc, t_init):
    particles = np.zeros((total_particles, 4))
    cell_width = lx / num_cells
    num_occupied_cells = num_cells // 2
    for i in range(num_occupied_cells):
        start_idx = i * ppc; end_idx = (i + 1) * ppc
        particles[start_idx:end_idx, 0] = i * cell_width + np.random.rand(ppc) * cell_width
    v_thermal_std = np.sqrt(KB * t_init / MASS_AR)
    particles[:, 1:4] = np.random.normal(0, v_thermal_std, (total_particles, 3))
    particles[:, 1:4] -= np.mean(particles[:, 1:4], axis=0)
    return particles

def sample_properties(particles_state, num_cells, cell_width, fnum, cell_vol, min_parts):
    density_profile = np.zeros(num_cells)
    velocity_profile = np.full(num_cells, np.nan)
    temp_profile = np.full(num_cells, np.nan)

    cell_indices = (particles_state[:, 0] / cell_width).astype(np.int64)
    cell_indices = np.clip(cell_indices, 0, num_cells - 1)

    sorted_indices = np.argsort(cell_indices)
    counts = np.bincount(cell_indices, minlength=num_cells)
    starts = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(counts[:-1])))

    for i in range(num_cells):
        num_in_cell = counts[i]
        if num_in_cell > 0:
            density_profile[i] = num_in_cell * fnum / cell_vol

        if num_in_cell >= min_parts:
            start_pos = starts[i]
            end_pos = start_pos + num_in_cell
            indices_in_cell_i = sorted_indices[start_pos:end_pos]

            cell_velocities = particles_state[indices_in_cell_i, 1:4]
            mean_vel_cell = np.mean(cell_velocities, axis=0)
            velocity_profile[i] = mean_vel_cell[0]

            thermal_vel_sq = np.sum((cell_velocities - mean_vel_cell)**2)
            temp_profile[i] = (MASS_AR * thermal_vel_sq) / (3 * KB * num_in_cell) if num_in_cell > 1 else 0.0

    return {'density': density_profile, 'velocity': velocity_profile, 'temperature': temp_profile}

if __name__ == "__main__":
    SIMULATION_PARAMS = {
        'LX': 1.0e-6, 'RHO_INIT': 1.78, 'T_INIT': 273.0,
        'NUM_CELLS_X': 200,
        'PARTICLES_PER_CELL_INIT': 10000,
        'TOTAL_TIME': 0.4e-9, 'DT': 5.0e-12,
    }
    NUM_ENSEMBLE_RUNS = 50

    print(f"--- شروع اجرای نهایی با مدل برخورد SBT بهینه شده (O(N)) (تعداد اجرا: {NUM_ENSEMBLE_RUNS}) ---")

    all_results = []
    start_time_total = time.time()
    for i in range(NUM_ENSEMBLE_RUNS):
        np.random.seed(int(time.time()) + i)
        single_run_history = run_dsmc_simulation(SIMULATION_PARAMS)
        all_results.append(single_run_history)
        print(f"--- اجرای {i+1}/{NUM_ENSEMBLE_RUNS} تمام شد ---")

    # ... (بخش میانگین‌گیری و رسم نمودار بدون تغییر) ...
    averaged_results = {}
    sample_times = sorted(all_results[0].keys())
    for t in sample_times:
        all_densities = np.array([run[t]['density'] for run in all_results if t in run])
        all_velocities = np.array([run[t]['velocity'] for run in all_results if t in run])
        all_temperatures = np.array([run[t]['temperature'] for run in all_results if t in run])
        averaged_results[t] = {
            'density': np.nanmean(all_densities, axis=0),
            'velocity': np.nanmean(all_velocities, axis=0),
            'temperature': np.nanmean(all_temperatures, axis=0),
        }

    cell_width = SIMULATION_PARAMS['LX'] / SIMULATION_PARAMS['NUM_CELLS_X']
    cell_centers = (np.arange(SIMULATION_PARAMS['NUM_CELLS_X']) + 0.5) * cell_width

    fig, axes = plt.subplots(3, 1, figsize=(12, 18), sharex=True)
    fig.suptitle(f"DSMC with Optimized SBT Collisions (N_runs={NUM_ENSEMBLE_RUNS}, N_p_cell={SIMULATION_PARAMS['PARTICLES_PER_CELL_INIT']})", fontsize=16)

    for t in sample_times:
        data = averaged_results[t]
        label_text = f't = {t*1e9:.2f} ns'
        line_style = '--' if t == 0.0 else '-'

        s_dens = pd.Series(data['density']).interpolate(method='linear')
        axes[0].plot(cell_centers, s_dens, linestyle=line_style, label=label_text)

        s_vel = pd.Series(data['velocity']).interpolate(method='linear').fillna(0)
        vel_smoothed = savgol_filter(s_vel.to_numpy(), window_length=21, polyorder=2)
        axes[1].plot(cell_centers, vel_smoothed, linestyle=line_style, label=label_text)

        s_temp = pd.Series(data['temperature']).interpolate(method='linear').bfill().ffill()
        temp_smoothed = savgol_filter(s_temp.to_numpy(), window_length=21, polyorder=2)
        axes[2].plot(cell_centers, temp_smoothed, linestyle=line_style, label=label_text)

    axes[0].set_ylabel('Number Density ($m^{-3}$)'); axes[0].set_title('Density Profile'); axes[0].grid(True, linestyle=':'); axes[0].legend()
    axes[1].set_ylabel('Bulk Velocity (m/s)'); axes[1].set_title('Bulk Velocity Profile (Smoothed)'); axes[1].grid(True, linestyle=':'); axes[1].legend()
    axes[2].set_ylabel('Temperature (K)'); axes[2].set_title('Temperature Profile (Smoothed - Optimized SBT)'); axes[2].grid(True, linestyle=':'); axes[2].legend()
    axes[2].set_xlabel('Position x (m)')

    axes[1].set_ylim(bottom=-50)
    axes[2].set_ylim(bottom=150, top=SIMULATION_PARAMS['T_INIT'] + 20)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig("dsmc_v17_1_optimized_sbt_result.png", dpi=300)
    plt.show()